In [ ]:
from pathlib import Path

KAGGLE_WORKING         = Path("/kaggle/working")
LORA_MERGE_DIR         = Path("/kaggle/input/datasets/maituananh511/finetune-vintern-44k/Vintern_Finetune_44k/work_dirs/internvl_chat_v2_0/Vintern_1B_v2_finetune_lora_viet_chart_vqa_merge")
PRETRAIN_DIR           = KAGGLE_WORKING / "Vintern/pretrained/Vintern-1B-v2"
SRC_DATASET            = "/kaggle/input/datasets/maituananh511/test-dataset-chart-vqa/vi_chart_dataset"
DST_DATASET            = str(KAGGLE_WORKING / "vi_chart_dataset")
VIETNAMESE_DATA_PATH   = Path("/kaggle/input/datasets/maituananh511/data-vietnamese/Data Vietnamese")
VIETNAMESE_IMAGES_PATH = VIETNAMESE_DATA_PATH / "images"
VIETNAMESE_JSONL_PATH  = VIETNAMESE_DATA_PATH / "viet_chart_vqa.jsonl"
LLAMA_DIR = Path("/kaggle/working/compare_models/llama-3.2-1b-instruct")

CHART_TEST_N = 500
VIETNAMESE_N = 200
EVAL_TOTAL   = CHART_TEST_N + VIETNAMESE_N

print("LORA_MERGE_DIR  exists:", LORA_MERGE_DIR.exists())
print("VIETNAMESE_DATA exists:", VIETNAMESE_DATA_PATH.exists())
print(f"Eval target: {EVAL_TOTAL} samples ({CHART_TEST_N} chart + {VIETNAMESE_N} vietnamese)")

In [ ]:
!pip install huggingface_hub==0.24.7

In [ ]:
!pip install -q transformers==4.44.2 --force-reinstall
!pip uninstall numpy -y
!pip install -q numpy==1.26.0
!pip install -q packaging ninja datasets timm einops peft deepspeed bitsandbytes decord gdown hf_transfer
# Verify versions
import importlib
import subprocess
result = subprocess.run(["pip", "show", "transformers", "huggingface_hub"], capture_output=True, text=True)
print(result.stdout)


In [ ]:
!pip install -q rouge_score evaluate underthesea bert-score
!pip install --upgrade nltk
!pip install -q numpy==2.0.0

In [ ]:
import subprocess, sys
subprocess.run(['pip', 'install', '-q', '--upgrade', 'huggingface_hub'], check=True)
for mod in list(sys.modules.keys()):
    if 'huggingface_hub' in mod:
        del sys.modules[mod]

from huggingface_hub import snapshot_download
import os

if not PRETRAIN_DIR.exists():
    print("⬇  Downloading Vintern-1B-v2 pretrain ...")
    snapshot_download(repo_id='5CD-AI/Vintern-1B-v2', local_dir=str(PRETRAIN_DIR))
    print(" Vintern-1B-v2 downloaded")
else:
    print(" Vintern-1B-v2 already exists")

In [ ]:
from huggingface_hub import snapshot_download, hf_hub_download
import requests, os

def check_internet(url="https://huggingface.co", timeout=5):
    try:
        requests.get(url, timeout=timeout)
        return True
    except Exception:
        return False

REPO_ID = 'unsloth/Llama-3.2-1B-Instruct'  

if LLAMA_DIR.exists() and any(LLAMA_DIR.iterdir()):
    print(" Llama-3.2-1B-Instruct already exists, skip download.")
else:
    LLAMA_DIR.mkdir(parents=True, exist_ok=True)
    if not check_internet():
        print(" Không có kết nối Internet.")
        print("👉 Notebook Settings → Internet → bật ON, rồi chạy lại.")
    else:
        print("⬇  Cách 1: snapshot_download ...")
        try:
            snapshot_download(
                repo_id=REPO_ID,
                local_dir=str(LLAMA_DIR),
                ignore_patterns=["*.msgpack", "*.h5", "flax_model*", "tf_model*"],
            )
            print(" Llama-3.2-1B-Instruct downloaded (cách 1)")
        except Exception as e1:
            print(f"    Cách 1 thất bại: {e1}")
            print("\n⬇  Cách 2: download từng file ...")
            REQUIRED_FILES = [
                "config.json", "tokenizer.json", "tokenizer_config.json",
                "special_tokens_map.json", "generation_config.json",
                "model.safetensors",
            ]
            for fname in REQUIRED_FILES:
                try:
                    hf_hub_download(repo_id=REPO_ID, filename=fname, local_dir=str(LLAMA_DIR))
                    print(f"   {fname}")
                except Exception as ef:
                    print(f"   {fname}: {ef}")

print(f"\nLLAMA_DIR = '{LLAMA_DIR}'")
print(f"Files found: {len(list(LLAMA_DIR.iterdir())) if LLAMA_DIR.exists() else 0}")

In [ ]:
from datasets import load_from_disk
from PIL import Image
import json, shutil, os

def is_dataset_complete(dst_path, src_path):
    """Kiểm tra dst có đủ file như src không."""
    if not os.path.exists(dst_path):
        return False
    src_files = set()
    for root, _, files in os.walk(src_path):
        for f in files:
            rel = os.path.relpath(os.path.join(root, f), src_path)
            src_files.add(rel)
    dst_files = set()
    for root, _, files in os.walk(dst_path):
        for f in files:
            rel = os.path.relpath(os.path.join(root, f), dst_path)
            dst_files.add(rel)
    missing = src_files - dst_files
    if missing:
        print(f"    Thiếu {len(missing)} files: {list(missing)[:5]} ...")
        return False
    return True

if is_dataset_complete(DST_DATASET, SRC_DATASET):
    print(" Dataset đã copy đầy đủ, skip.")
else:
    if os.path.exists(DST_DATASET):
        print("🗑  Xóa bản copy cũ bị thiếu file ...")
        shutil.rmtree(DST_DATASET)
    print("📋 Copying dataset to working dir ...")
    shutil.copytree(SRC_DATASET, DST_DATASET)
    print(" Copy xong.")

vi_chart_dataset = load_from_disk(DST_DATASET)
print(vi_chart_dataset)

vietnamese_records = []
with open(VIETNAMESE_JSONL_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            vietnamese_records.append(json.loads(line))
print(f"Vietnamese records: {len(vietnamese_records)} loaded")

In [ ]:
from PIL import Image

def normalize_turn(turn):
    if isinstance(turn, str):
        return {'role': 'assistant', 'content': turn}
    if isinstance(turn, dict):
        role = turn.get('role') or turn.get('from', '')
        if role in ('human', 'user'):      role = 'user'
        elif role in ('gpt', 'assistant'): role = 'assistant'
        for rk in ('assistant', 'user', 'human', 'gpt'):
            if rk in turn and 'content' not in turn and 'role' not in turn:
                role = 'assistant' if rk in ('assistant', 'gpt') else 'user'
                return {'role': role, 'content': str(turn[rk])}
        content = str(turn.get('content') or turn.get('value', ''))
        return {'role': role, 'content': content}
    return {'role': 'assistant', 'content': str(turn)}

chart_test_raw   = vi_chart_dataset['test']
chart_n          = min(CHART_TEST_N, len(chart_test_raw))
chart_test_items = [chart_test_raw[i] for i in range(chart_n)]
print(f" vi_chart test   : {chart_n} samples")

vn_test_items = []
for record in reversed(vietnamese_records):
    if len(vn_test_items) >= VIETNAMESE_N:
        break
    img_path = VIETNAMESE_IMAGES_PATH / record['image']
    try:
        image = Image.open(img_path).convert('RGB')
    except Exception as e:
        print(f"Warning: cannot open {img_path}: {e}")
        continue
    convs = [normalize_turn(t) for t in record['conversations']]
    pairs = [(convs[i], convs[i+1]) for i in range(0, len(convs)-1, 2)]
    for idx, (q, a) in enumerate(pairs):
        if len(vn_test_items) >= VIETNAMESE_N:
            break
        rid = record['id'] if len(pairs) == 1 else f"{record['id']}_q{idx}"
        vn_test_items.append({'id': rid, 'image': image, 'conversations': [q, a]})

print(f" vietnamese test  : {len(vn_test_items)} samples")

eval_dataset = chart_test_items + vn_test_items
print(f"\n EVAL DATASET: {len(eval_dataset)} samples ({chart_n} chart + {len(vn_test_items)} vietnamese)")
assert len(eval_dataset) == EVAL_TOTAL, f" Mong đợi {EVAL_TOTAL}, thực tế {len(eval_dataset)}"
print(f" Đúng {EVAL_TOTAL} samples!")

In [ ]:
import torch
import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode
from PIL import Image

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

def build_transform(input_size):
    return T.Compose([
        T.Lambda(lambda img: img.convert('RGB') if img.mode != 'RGB' else img),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    ])

def find_closest_aspect_ratio(aspect_ratio, target_ratios, width, height, image_size):
    best_ratio_diff = float('inf')
    best_ratio = (1, 1)
    area = width * height
    for ratio in target_ratios:
        target_aspect_ratio = ratio[0] / ratio[1]
        ratio_diff = abs(aspect_ratio - target_aspect_ratio)
        if ratio_diff < best_ratio_diff:
            best_ratio_diff = ratio_diff
            best_ratio = ratio
        elif ratio_diff == best_ratio_diff:
            if area > 0.5 * image_size * image_size * ratio[0] * ratio[1]:
                best_ratio = ratio
    return best_ratio

def dynamic_preprocess(image, min_num=1, max_num=12, image_size=448, use_thumbnail=False):
    orig_width, orig_height = image.size
    aspect_ratio = orig_width / orig_height
    target_ratios = set(
        (i, j) for n in range(min_num, max_num + 1)
        for i in range(1, n + 1) for j in range(1, n + 1)
        if min_num <= i * j <= max_num
    )
    target_ratios = sorted(target_ratios, key=lambda x: x[0] * x[1])
    target_aspect_ratio = find_closest_aspect_ratio(aspect_ratio, target_ratios, orig_width, orig_height, image_size)
    target_width  = image_size * target_aspect_ratio[0]
    target_height = image_size * target_aspect_ratio[1]
    blocks = target_aspect_ratio[0] * target_aspect_ratio[1]
    resized_img = image.resize((target_width, target_height))
    processed_images = []
    for i in range(blocks):
        box = (
            (i % (target_width  // image_size)) * image_size,
            (i // (target_width // image_size)) * image_size,
            ((i % (target_width  // image_size)) + 1) * image_size,
            ((i // (target_width // image_size)) + 1) * image_size
        )
        processed_images.append(resized_img.crop(box))
    if use_thumbnail and len(processed_images) != 1:
        processed_images.append(image.resize((image_size, image_size)))
    return processed_images

def load_image(image_file, input_size=448, max_num=12):
    image = Image.open(image_file).convert('RGB') if isinstance(image_file, str) else image_file
    image = image.resize((input_size, input_size))
    transform = build_transform(input_size)
    images = dynamic_preprocess(image, image_size=input_size, use_thumbnail=True, max_num=max_num)
    pixel_values = torch.stack([transform(img) for img in images])
    return pixel_values

print(" Image utils ready")

In [ ]:
import importlib, sys, os, torch, nltk, pandas as pd
for key in list(sys.modules.keys()):
    if 'nltk' in key:
        del sys.modules[key]

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score as nltk_meteor
from rouge_score import rouge_scorer
from tqdm import tqdm
from underthesea import word_tokenize

def setup_nltk_data(path='/usr/share/nltk_data'):
    os.makedirs(path, exist_ok=True)
    nltk.data.path.append(path)
    for pkg in ['punkt', 'wordnet', 'omw-1.4']:
        nltk.download(pkg, download_dir=path, quiet=True)

def compute_bertscore(responses, references):
    try:
        import bert_score as bs_lib
        _, _, F1 = bs_lib.score(responses, references, lang='vi', verbose=False,
                                rescale_with_baseline=False)
        return F1.tolist()
    except Exception as e:
        print(f'BERTScore failed: {e}')
        return [0.0] * len(responses)

def _compute_scores(results, bleu_scores, meteor_scores, rouge_scores, all_responses, all_references):
    bert_f1 = compute_bertscore(all_responses, all_references) if all_responses else []
    for j, row in enumerate(results):
        row['bertscore'] = bert_f1[j] if j < len(bert_f1) else 0.0
    n = max(len(results), 1)
    avg = {
        'bleu':      sum(bleu_scores) / n,
        'meteor':    sum(meteor_scores) / n,
        'rouge1':    sum(rouge_scores['rouge1']) / n,
        'rouge2':    sum(rouge_scores['rouge2']) / n,
        'rougeL':    sum(rouge_scores['rougeL']) / n,
        'bertscore': sum(r['bertscore'] for r in results) / n,
    }
    return pd.DataFrame(results), avg

def evaluate_vintern(eval_dataset, model, tokenizer, model_name='Vintern', debug=False):
    """Evaluate Vintern multimodal model (vision + text)."""
    setup_nltk_data()
    bleu_scores, meteor_scores = [], []
    rouge_scores = {'rouge1': [], 'rouge2': [], 'rougeL': []}
    all_responses, all_references, results = [], [], []
    scorer   = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    smoothie = SmoothingFunction().method1
    gen_cfg  = dict(max_new_tokens=1024, do_sample=False, num_beams=3, repetition_penalty=2.0)

    for item in tqdm(eval_dataset, desc=f'[{model_name}]'):
        if not all(k in item for k in ['id', 'image', 'conversations']): continue
        image_id     = item['id']
        question_txt = str(item['conversations'][0]['content'])
        ground_truth = str(item['conversations'][1]['content'])
        try:
            pv       = load_image(item['image'], max_num=12).to(torch.bfloat16).cuda()
            response = model.chat(tokenizer, pv, f'<image>\n{question_txt}', gen_cfg)
        except Exception as e:
            if debug: print(f'Error [{image_id}]: {e}')
            response = ''

        ref = word_tokenize(ground_truth, format='text').split()
        hyp = word_tokenize(response, format='text').split() if response else ['']
        bleu   = sentence_bleu([ref], hyp, smoothing_function=smoothie)
        meteor = float(nltk_meteor([ref], hyp))
        r      = scorer.score(ground_truth, response)

        bleu_scores.append(bleu); meteor_scores.append(meteor)
        for k in ['rouge1', 'rouge2', 'rougeL']: rouge_scores[k].append(r[k].fmeasure)
        all_responses.append(response); all_references.append(ground_truth)
        results.append({'id': image_id, 'question': question_txt, 'ground_truth': ground_truth,
                        'response': response, 'bleu': bleu, 'meteor': meteor,
                        'rouge1': r['rouge1'].fmeasure, 'rouge2': r['rouge2'].fmeasure,
                        'rougeL': r['rougeL'].fmeasure})

    return _compute_scores(results, bleu_scores, meteor_scores, rouge_scores, all_responses, all_references)

def evaluate_text_model(eval_dataset, model, tokenizer, model_name='TextModel', debug=False):
    """Evaluate text-only model (không có ảnh, chỉ dùng câu hỏi)."""
    setup_nltk_data()
    bleu_scores, meteor_scores = [], []
    rouge_scores = {'rouge1': [], 'rouge2': [], 'rougeL': []}
    all_responses, all_references, results = [], [], []
    scorer   = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    smoothie = SmoothingFunction().method1

    for item in tqdm(eval_dataset, desc=f'[{model_name}]'):
        if not all(k in item for k in ['id', 'conversations']): continue
        image_id     = item['id']
        question_txt = str(item['conversations'][0]['content'])
        ground_truth = str(item['conversations'][1]['content'])
        prompt = (
            "Bạn là trợ lý AI thông minh. Hãy trả lời câu hỏi sau về biểu đồ bằng tiếng Việt.\n"
            f"Câu hỏi: {question_txt}\nTrả lời:"
        )
        try:
            inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512).to(model.device)
            with torch.no_grad():
                out = model.generate(**inputs, max_new_tokens=256, do_sample=False,
                                     pad_token_id=tokenizer.eos_token_id)
            response = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
        except Exception as e:
            if debug: print(f'Error [{image_id}]: {e}')
            response = ''

        ref = word_tokenize(ground_truth, format='text').split()
        hyp = word_tokenize(response, format='text').split() if response else ['']
        bleu   = sentence_bleu([ref], hyp, smoothing_function=smoothie)
        meteor = float(nltk_meteor([ref], hyp))
        r      = scorer.score(ground_truth, response)

        bleu_scores.append(bleu); meteor_scores.append(meteor)
        for k in ['rouge1', 'rouge2', 'rougeL']: rouge_scores[k].append(r[k].fmeasure)
        all_responses.append(response); all_references.append(ground_truth)
        results.append({'id': image_id, 'question': question_txt, 'ground_truth': ground_truth,
                        'response': response, 'bleu': bleu, 'meteor': meteor,
                        'rouge1': r['rouge1'].fmeasure, 'rouge2': r['rouge2'].fmeasure,
                        'rougeL': r['rougeL'].fmeasure})

    return _compute_scores(results, bleu_scores, meteor_scores, rouge_scores, all_responses, all_references)

print(" Evaluate functions ready")

In [ ]:
from transformers import AutoModel, AutoTokenizer
import torch
import sys
import types

# Tạo mock module có __spec__ hợp lệ
def make_mock_module(name):
    mod = types.ModuleType(name)
    mod.__spec__ = types.SimpleNamespace(name=name)
    return mod

for mod_name in [
    'flash_attn',
    'flash_attn.bert_padding',
    'flash_attn.flash_attn_interface',
    'flash_attn.flash_attn_funcs',
]:
    sys.modules[mod_name] = make_mock_module(mod_name)

print("📦 Loading Vintern LoRA Fine-tuned ...")
lora_model = AutoModel.from_pretrained(
    str(LORA_MERGE_DIR),
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
    attn_implementation="eager",
).eval().cuda()
lora_tokenizer = AutoTokenizer.from_pretrained(str(LORA_MERGE_DIR), trust_remote_code=True, use_fast=False)
print(" Vintern LoRA loaded")

In [ ]:
import os

print(" Kiểm tra LLAMA_DIR:", LLAMA_DIR)
if os.path.exists(LLAMA_DIR):
    files = os.listdir(LLAMA_DIR)
    print(f"  Files ({len(files)}): {files}")
    has_config  = 'config.json' in files
    has_weights = any(f.endswith('.safetensors') for f in files)
    print(f"  config.json   : {'' if has_config  else ' THIẾU'}")
    print(f"  *.safetensors : {'' if has_weights else ' THIẾU'}")
    if not has_config or not has_weights:
        print("\n  Model chưa download đủ — chạy lại cell Download Llama trước!")
else:
    print("   Folder không tồn tại — chưa download!")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer as HFTokenizer
import torch, gc

all_results = {}

print('=' * 60)
print(f' Evaluating: Vintern-LoRA-Finetuned  [{len(eval_dataset)} samples]')
df_vintern, avg_vintern = evaluate_vintern(eval_dataset, lora_model, lora_tokenizer,
                                           model_name='Vintern-LoRA-Finetuned', debug=True)
all_results['Vintern-LoRA-Finetuned'] = (df_vintern, avg_vintern)
print('  →', {k: round(v, 4) for k, v in avg_vintern.items()})

# Lưu CSV từng model để debug nếu bị crash ở bước sau
df_vintern.to_csv('/kaggle/working/debug_vintern_lora.csv', index=False, encoding='utf-8')
print("   Saved debug_vintern_lora.csv")

# Unload Vintern
del lora_model, lora_tokenizer
torch.cuda.empty_cache(); gc.collect()
print('\n🗑  Vintern unloaded — VRAM freed')

print('\n' + '=' * 60)
print(f' Evaluating: Llama-3.2-1B-Instruct  [{len(eval_dataset)} samples]')
try:
    tok_l = HFTokenizer.from_pretrained(LLAMA_DIR, trust_remote_code=True)
    if tok_l.pad_token is None: tok_l.pad_token = tok_l.eos_token
    mdl_l = AutoModelForCausalLM.from_pretrained(
        LLAMA_DIR,
        torch_dtype=torch.bfloat16, low_cpu_mem_usage=True,
        device_map='auto', trust_remote_code=True
    ).eval()
    df_l, avg_l = evaluate_text_model(eval_dataset, mdl_l, tok_l,
                                       model_name='Llama-3.2-1B-Instruct', debug=True)
    all_results['Llama-3.2-1B-Instruct'] = (df_l, avg_l)
    print('  →', {k: round(v, 4) for k, v in avg_l.items()})
    df_l.to_csv('/kaggle/working/debug_llama32_1b.csv', index=False, encoding='utf-8')
    print("   Saved debug_llama32_1b.csv")
    del mdl_l, tok_l; torch.cuda.empty_cache(); gc.collect()
    print('  🗑  Llama-3.2-1B-Instruct unloaded')
except Exception as e:
    print(f'   Llama-3.2-1B-Instruct failed: {e}')

print(f'\n Done! {len(eval_dataset)} samples × {len(all_results)} models')

In [ ]:
import subprocess, importlib

def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout[-500:] if r.stdout else '')
    if r.stderr and 'error' in r.stderr.lower():
        print('STDERR:', r.stderr[-300:])
    return r.returncode

print("📦 Upgrading transformers + tokenizers...")
run("pip install -q 'transformers>=4.45.0' 'tokenizers>=0.20.0' --upgrade")

import tokenizers, transformers
print(f"\n transformers : {transformers.__version__}")
print(f" tokenizers   : {tokenizers.__version__}")

from tokenizers import Tokenizer as HFRustTokenizer
import os
TOK_PATH = os.path.join(str(LLAMA_DIR), 'tokenizer.json')
try:
    _ = HFRustTokenizer.from_file(TOK_PATH)
    print("\n tokenizer.json load OK — có thể load model bình thường!")
except Exception as e:
    print(f"\n Vẫn lỗi: {e}")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer as HFTokenizer
import torch, gc, os

config_path = os.path.join(LLAMA_DIR, 'config.json')
if not os.path.isfile(config_path):
    print(f" Không tìm thấy config.json trong: {LLAMA_DIR}")
    print(f"   Files hiện có: {os.listdir(LLAMA_DIR) if os.path.exists(LLAMA_DIR) else 'folder không tồn tại'}")
    print("\n👉 Hãy chạy lại cell '4. Download Llama-3.2-1B-Instruct' trước.")
    raise FileNotFoundError(f"config.json not found in {LLAMA_DIR}")

print(f" Loading Llama-3.2-1B-Instruct từ: {LLAMA_DIR}")
tok_l = HFTokenizer.from_pretrained(LLAMA_DIR, trust_remote_code=True, local_files_only=True)
if tok_l.pad_token is None:
    tok_l.pad_token = tok_l.eos_token

mdl_l = AutoModelForCausalLM.from_pretrained(
    LLAMA_DIR,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    device_map='auto',
    trust_remote_code=True,
    local_files_only=True,
).eval()
print(" Llama-3.2-1B-Instruct loaded")

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score as nltk_meteor
from rouge_score import rouge_scorer
from tqdm import tqdm
from underthesea import word_tokenize

def evaluate_text_model(eval_dataset, model, tokenizer, model_name='TextModel', debug=False):
    setup_nltk_data()
    bleu_scores, meteor_scores = [], []
    rouge_scores = {'rouge1': [], 'rouge2': [], 'rougeL': []}
    all_responses, all_references, results = [], [], []
    scorer   = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    smoothie = SmoothingFunction().method1

    for item in tqdm(eval_dataset, desc=f'[{model_name}]'):
        if not all(k in item for k in ['id', 'conversations']): continue
        image_id     = item['id']
        question_txt = str(item['conversations'][0]['content'])
        ground_truth = str(item['conversations'][1]['content'])
        prompt = (
            "Bạn là trợ lý AI thông minh. Hãy trả lời câu hỏi sau về biểu đồ bằng tiếng Việt.\n"
            f"Câu hỏi: {question_txt}\nTrả lời:"
        )
        try:
            inputs = tokenizer(
                prompt,
                return_tensors='pt',
                truncation=True,
                max_length=512,
            ).to(model.device)

            input_len = inputs['input_ids'].shape[1]

            with torch.no_grad():
                out = model.generate(
                    **inputs,
                    max_new_tokens=256,
                    do_sample=False,
                    temperature=None,       
                    top_p=None,             
                    pad_token_id=tokenizer.eos_token_id,
                )
            response = tokenizer.decode(
                out[0][input_len:], skip_special_tokens=True
            ).strip()
        except Exception as e:
            if debug: print(f'Error [{image_id}]: {e}')
            response = ''

        ref = word_tokenize(ground_truth, format='text').split()
        hyp = word_tokenize(response, format='text').split() if response else ['']
        bleu   = sentence_bleu([ref], hyp, smoothing_function=smoothie)
        meteor = float(nltk_meteor([ref], hyp))
        r      = scorer.score(ground_truth, response)

        bleu_scores.append(bleu); meteor_scores.append(meteor)
        for k in ['rouge1', 'rouge2', 'rougeL']: rouge_scores[k].append(r[k].fmeasure)
        all_responses.append(response); all_references.append(ground_truth)
        results.append({
            'id': image_id, 'question': question_txt,
            'ground_truth': ground_truth, 'response': response,
            'bleu': bleu, 'meteor': meteor,
            'rouge1': r['rouge1'].fmeasure, 'rouge2': r['rouge2'].fmeasure,
            'rougeL': r['rougeL'].fmeasure,
        })

    return _compute_scores(results, bleu_scores, meteor_scores, rouge_scores, all_responses, all_references)

print(f"\n Evaluating Llama-3.2-1B-Instruct [{len(eval_dataset)} samples] ...")
df_l, avg_l = evaluate_text_model(eval_dataset, mdl_l, tok_l,
                                   model_name='Llama-3.2-1B-Instruct', debug=True)

print('\n Kết quả Llama-3.2-1B-Instruct:')
for k, v in avg_l.items():
    print(f"   {k:12s}: {v:.4f}")

df_l.to_csv('/kaggle/working/debug_llama32_1b.csv', index=False, encoding='utf-8')
print("\n Saved: debug_llama32_1b.csv")

if 'all_results' not in dir():
    all_results = {}
all_results['Llama-3.2-1B-Instruct'] = (df_l, avg_l)
print(f" all_results hiện có: {list(all_results.keys())}")

del mdl_l, tok_l
torch.cuda.empty_cache(); gc.collect()
print('🗑  Llama-3.2-1B-Instruct unloaded — VRAM freed')

In [ ]:
import pandas as pd, matplotlib.pyplot as plt, numpy as np

metrics_keys = ['bleu', 'meteor', 'rouge1', 'rouge2', 'rougeL', 'bertscore']
rows = []
for mname, (df, avg) in all_results.items():
    row = {'Model': mname}
    for m in metrics_keys:
        row[m.upper()] = round(avg.get(m, 0.0), 4)
    rows.append(row)

summary_df = pd.DataFrame(rows).set_index('Model')

def highlight_best(s):  return ['background-color: #d4edda; font-weight: bold' if v == s.max() else '' for v in s]
def highlight_worst(s): return ['background-color: #f8d7da' if v == s.min() else '' for v in s]

n_samples = len(eval_dataset)
print(f'\n KẾT QUẢ ({n_samples} samples | Ô xanh = cao nhất | Ô đỏ = thấp nhất)\n')
display(summary_df.style.apply(highlight_best).apply(highlight_worst))

csv_path = '/kaggle/working/eval_vintern_lora_vs_llama32_1b.csv'
summary_df.reset_index().to_csv(csv_path, index=False, encoding='utf-8')
print(f'\n Saved CSV: eval_vintern_lora_vs_llama32_1b.csv')

n = len(all_results)
x = np.arange(len(metrics_keys))
width = 0.8 / n
colors = ['#4878cf', '#d65f5f']

fig, ax = plt.subplots(figsize=(13, 5))
for idx, (mname, (_, avg)) in enumerate(all_results.items()):
    offset = idx * width - (n - 1) * width / 2
    vals   = [avg.get(m, 0.0) for m in metrics_keys]
    bars   = ax.bar(x + offset, vals, width, label=mname, color=colors[idx % len(colors)])
    ax.bar_label(bars, fmt='%.3f', padding=2, fontsize=8, rotation=90)

ax.set_xticks(x)
ax.set_xticklabels([m.upper() for m in metrics_keys], fontsize=10)
ax.set_ylabel('Score'); ax.set_ylim(0, 1.2)
ax.set_title(f'Vintern-LoRA-Finetuned vs Llama-3.2-1B-Instruct  ({n_samples} samples)', fontsize=12)
ax.legend(fontsize=10); plt.tight_layout()

img_path = '/kaggle/working/eval_vintern_lora_vs_llama32_1b.png'
plt.savefig(img_path, dpi=150); plt.show()
print(f' Saved chart: eval_vintern_lora_vs_llama32_1b.png')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

metrics_keys = ['bleu', 'meteor', 'rouge1', 'rouge2', 'rougeL', 'bertscore']


current_model_name, (current_df, current_avg) = list(all_results.items())[0]

other_csv_path = '/kaggle/working/debug_vintern_lora.csv'
other_df = pd.read_csv(other_csv_path)

other_df.columns = [c.strip().lower() for c in other_df.columns]

other_avg = {}
for m in metrics_keys:
    if m.lower() in other_df.columns:
        other_avg[m] = other_df[m.lower()].mean()
    else:
        other_avg[m] = 0.0
        print(f' Không tìm thấy cột "{m}" trong CSV, gán = 0.0')

other_model_name = 'Vintern-LoRA (CSV)'

rows = []

row1 = {'Model': current_model_name}
for m in metrics_keys:
    row1[m.upper()] = round(current_avg.get(m, 0.0), 4)
rows.append(row1)

row2 = {'Model': other_model_name}
for m in metrics_keys:
    row2[m.upper()] = round(other_avg.get(m, 0.0), 4)
rows.append(row2)

summary_df = pd.DataFrame(rows).set_index('Model')

def highlight_best(s):
    return ['background-color: #d4edda; font-weight: bold' if v == s.max() else '' for v in s]

def highlight_worst(s):
    return ['background-color: #f8d7da' if v == s.min() else '' for v in s]

n_samples_current = len(current_df) if current_df is not None else 'Unknown'
n_samples_other = len(other_df)

print(f'\n KẾT QUẢ SO SÁNH 2 MODEL')
print(f'- {current_model_name}: {n_samples_current} samples')
print(f'- {other_model_name}: {n_samples_other} samples')
print(f'Ô xanh = cao nhất | Ô đỏ = thấp nhất\n')

display(summary_df.style.apply(highlight_best).apply(highlight_worst))

csv_path = '/kaggle/working/compare_2_models_summary.csv'
summary_df.reset_index().to_csv(csv_path, index=False, encoding='utf-8')
print(f' Saved CSV: {csv_path}')

models_data = [
    (current_model_name, current_avg),
    (other_model_name, other_avg)
]

n = len(models_data)  
x = np.arange(len(metrics_keys))
width = 0.35
colors = ['#4878cf', '#d65f5f']

fig, ax = plt.subplots(figsize=(13, 5))

for idx, (mname, avg) in enumerate(models_data):
    offset = (idx - (n - 1) / 2) * width
    vals = [avg.get(m, 0.0) for m in metrics_keys]
    bars = ax.bar(x + offset, vals, width, label=mname, color=colors[idx % len(colors)])
    ax.bar_label(bars, fmt='%.3f', padding=2, fontsize=8, rotation=90)

ax.set_xticks(x)
ax.set_xticklabels([m.upper() for m in metrics_keys], fontsize=10)
ax.set_ylabel('Score')
ax.set_ylim(0, 1.2)
ax.set_title('Comparison of 2 Models', fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout()

img_path = '/kaggle/working/compare_2_models_chart.png'
plt.savefig(img_path, dpi=150)
plt.show()

print(f' Saved chart: {img_path}')